<a href="https://colab.research.google.com/github/Spilnyk/Blender-Addon-Photogrammetry-Importer/blob/master/southampton_computational_bioacoustics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

.[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jscanass/BIOS0032_AI4Environment/blob/main/BioacousticsAI/Notebook.ipynb)

# Computational Bioacoustics

In this notebook, you will explore how computer audition is applied to ecology.
We will focus on automated animal detection, learning how to interpret and evaluate model outputs using BatDetect2 as a practical case study.

If you are new to Google Colab, you might find it helpful to review the [Colab Features Overview](https://colab.research.google.com/notebooks/basic_features_overview.ipynb) before getting started.

**Note**: While this notebook includes a lot of Python code, you don't need to write or modify any of it.
You will just need to execute the code blocks as you go to see the results.
Feel free to focus entirely on the concepts and outputs; the underlying code is there in case you want to see how it works in practice.

## Contents

1. [Acoustic Ecology](#1-acoustic-ecology)
2. [Audio Analysis](#2-audio-analysis)
3. [Sound Event Detection](#3-sound-event-detection)

## 1. Acoustic Ecology

### 1.1 Computer Audition

Computer audition is the field of research that deals with the automatic analysis of audio signals.
It intersects with many other fields, including machine learning, signal processing, and computer vision.

**What does a computer hear**?

- Audio files are a sequence of numbers representing the amplitude of the sound wave (pressure level) at a given time.

- The number of samples per second is called the **sampling rate**.

<img alt="audio and sampling rate" width="600"
src="https://cdn.shopify.com/s/files/1/1169/2482/files/Sampling_Rate_Cover_image.jpg?v=1654170259"></img>

**What tasks can we do with computer audition?**

Computer audition is used in a wide range of applications, including:

- Speech recognition: Siri, Alexa, Google Assistant
- Music information retrieval: Spotify, Shazam
- Audio classification: What is sounding in this audio?
- Sound event detection: Transcription of audio into a sequence of events.

<img alt="Sound event detection" width="400"
src="http://d33wubrfki0l68.cloudfront.net/508a62f305652e6d9af853c65ab33ae9900ff38e/17a88/images/tasks/challenge2016/task3_overview.png"></img>

> Taken from the paper: Mesaros, A., Heittola, T., Diment, A., Elizalde, B., Shah, A., Vincent, E.,
> ... & Virtanen, T. (2017, November). DCASE 2017 challenge setup: Tasks, datasets and baseline
> system. In DCASE 2017-Workshop on Detection and Classification of Acoustic Scenes and Events.

Recently, deep learning has taken over the field of computer audition and is being used to solve many of the above tasks.

### 1.2 Data Collection

**Acoustic sensors** can be used to collect field recordings of animal sounds.

Usually, these sensors are deployed statically in the field for a long periods of time and record sounds continuously.
This is called **passive acoustic monitoring**.

<img alt="passive acoustic monitoring" width="400"
src="https://wittmann-tours.de/wp-content/uploads/2018/06/AudioMoth.jpg"></img>

Alternatively, recordings are actively directed towards a specific animal species or sound events.

<img alt="active recording" width="400"
src="https://s3.amazonaws.com/cdn.freshdesk.com/data/helpdesk/attachments/production/48032687175/original/xjI7Dy3Q9kaCZinr5vf4ksNxQbjK13Yv3A.jpg?1584552543"></img>

> Taken from the Macaulay Library blog post: [Sound recording > tips](https://support.ebird.org/en/support/solutions/articles/48001064298-sound-recording-tips).

### 1.3 Acoustics for Ecology

The sound at a site is a reflection of the species present in the area and other environmental factors.

<img alt="composition of acoustic space" width="500"
src="https://media.springernature.com/full/springer-static/image/art%3A10.1007%2Fs12304-017-9288-5/MediaObjects/12304_2017_9288_Fig1_HTML.gif?as=webp"></img>

> Taken from the paper: Mullet, T.C., Farina, A. & Gage, S.H. The Acoustic
> Habitat Hypothesis: An Ecoacoustics Perspective on Species Habitat Selection.
> Biosemiotics 10, 319–336 (2017). https://doi.org/10.1007/s12304-017-9288-5

If we could link sounds to the species or individuals that produced them, we could use this information to study and monitor the biodiversity of an area.

Acoustic sensors produce a lot of data, and it is not always easy to analyse.
Can we use computer audition to help us?

In this notebook we will explore the task of **animal sound detection** and **species classification**, using both manual and automated methods.

## 2. Audio Analysis

### 2.1 Setup

#### 2.1.1 Enable GPU Runtime

Go to `Runtime` -> `Change runtime type` and select `GPU` as the hardware accelerator.

#### 2.1.2 Run the setup script

Please run the cell bellow to donwload the data you will be using in this notebook as well as to install the missing dependencies.

In [ ]:
! wget -q -O - https://raw.githubusercontent.com/jscanass/BIOS0032_AI4Environment/refs/heads/main/BioacousticsAI/setup.sh | bash

#### 2.1.3 Import Dependencies

All the required dependencies for the notebook are in the cell below.
Please run before anything else.
If for any reason the runtime is restarted you will need to run it again.

In [ ]:
from pathlib import Path
from time import perf_counter

import audio_utils
import evaluation_utils
import ipywidgets as widgets
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotting_utils
import seaborn as sns
import tensorflow_hub as hub
from batdetect2 import api
from IPython.display import Audio, display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm
from umap import UMAP

### 2.2 Analysis

- Most of the time, we are not interested in all the sounds in a recording, but only in the sounds of a specific animal species.

- Acoustic sensors will indiscriminately record all sounds in the environment, including those of animals, wind, rain, _etc._ Although some recorders can be triggered by a specific sound, this is not always the case.

- Passive acoustic monitoring produces many hours of recordings, and it's hard to identify and explore the sounds of interest.

While not as developed as in **computer vision**, there are some tools for automatically **detecting animal sounds** in recordings.
Here we will explore a few of them.

But first we need to understand how to visualise and annotate sounds.

### 2.3 Animal Sounds Visualisation

- While we can listen to the sounds in a recording, it is often easier to visualise them.

- This is especially true when we want to compare sounds from different recordings or navigate quickly without the need to listen.

- We can use waveplots and spectrograms to visualise the sounds in a recording.

Now we will load a dataset of animal recordings provided by [Avisoft](https://www.avisoft.com/animal-sounds/) and visualise them.

In [ ]:
DATA_DIR = Path.cwd() / "data"

In [ ]:
AVISOFT_AUDIO_DIR = DATA_DIR / "avisoft" / "audio"
AVISOFT_METADATA_FILE = DATA_DIR / "avisoft" / "avisoft_metadata.csv"

In [ ]:
# Load the metadata dataframe
avisoft = pd.read_csv(AVISOFT_METADATA_FILE)

In [ ]:
# Print first few rows
avisoft.head()

In [ ]:
# select a random file from the dataset
random_recording = avisoft.sample(n=1).iloc[0]

# read the audio file and import it as a numpy array
wav, samplerate = librosa.load(AVISOFT_AUDIO_DIR / random_recording.wav, sr=None)

# Compute the duration of audio
num_samples = len(wav)  # Number of samples taken by the recorder
duration = num_samples / samplerate

# Get name of animal
animal_name = random_recording.english_name

print(f"File selected = {random_recording.wav}")
print(f"Samplerate = {samplerate} Hz")
print(f"Duration = {duration:.2f} s")
print(f"Species = {animal_name}")

Let us first listen to the audio:

In [ ]:
Audio(data=wav, rate=samplerate)

In [ ]:
# Create plot of the waveform
times = np.linspace(0, duration, num_samples)
plt.figure(figsize=(10, 3))
plt.plot(times, wav)
plt.xlabel("time (s)")
plt.title(f"Waveform of {animal_name} sound")

The **waveform** gives us a visual representation of the sound amplitude over time.

However, if there are multiple simultaneous sounds in the recording, it can be hard to see each individual sound.

We can use a **spectrogram** to decompose the sound into **frequencies** and visualise them as a 2D image.

In [ ]:
# Compute the spectrogram with the short time fourier transform (STFT)
spectrogram = np.abs(librosa.stft(wav))

# Amplitude is best represented in logarithmic scale (decibels)
db_spectrogram = librosa.amplitude_to_db(spectrogram, ref=np.max)

In [ ]:
# Create plot of spectrogram
num_freq_bins, num_time_bins = db_spectrogram.shape
times = np.linspace(0, duration, num_time_bins)
freqs = np.linspace(0, samplerate / 2, num_freq_bins)

plt.figure(figsize=(10, 4))
plt.pcolormesh(times, freqs, db_spectrogram, cmap="magma")
plt.colorbar()
plt.title(f"Spectrogram of {random_recording.english_name} sound")
plt.xlabel("time (s)")
plt.ylabel("freq (Hz)")

The sounds produced by animals can be very different from each other.
The transformation used to create the spectrogram, called the **short-time Fourier transform** (STFT), will highlight different features of the sound depending on the parameters used.

---

🖌️ Research what the STFT is and how its parameters affect the spectrogram.
In particular, try to understand the effect of the **window size** and the **hop size** or **overlap**.

The **Fourier transform** is a method for breaking a time series into its constituent frequencies.
[Click here](https://www.youtube.com/watch?v=spUNpyF58BY) if you are interested in an intuitive and visual explanation of the Fourier transform.

For the **short-time Fourier transform** the audio is broken up into **windows** (or **chunks** or **frames**), which usually overlap each other.
Each **window** is Fourier transformed, and the result is added to a matrix, which records magnitude for each point in time and frequency.

![short time fourier transform](https://www.mdpi.com/applsci/applsci-10-07208/article_deploy/html/images/applsci-10-07208-g001-550.jpg)

In order to compute a **STFT** of a signal you must select a `window_length` (or `n_fft`) and a `hop_length` (or `overlap`) to determine how to break up the signal into **windows**.

- The `hop_size` controls the temporal resolution, or the minimum interval at which you can detect changes in sound.
  If `hop_length = 128` then any transient sounds of length less than 128 samples will be hard to detect.

- The `window_length` controls the frequecy resolution.
  With larger `window_length` it is possible to distinguish between closer frequencies.

- Selecting a high/low value for `window_length` will produce spectrograms with many/few frequency bins.

- Similarly, a high/low value for `hop_length` will produce spectrograms with many/freq time bins.

Here you can visualise sounds from different species and see how the STFT parameters affect the spectrogram.

In [ ]:
# @title Interactive spectrogram of animal sounds

# @markdown Select the file you wish to visualise. Modify the spectrogram parameters to see its effect on the spectrogram. Change the reproduction speed for interesting effects!

# Select some varied sounds from avisoft dataset
examples = [
    (row.english_name, AVISOFT_AUDIO_DIR / row.wav)
    # select one random recording per taxonomic group
    for row in avisoft.groupby("order").sample(n=1).itertuples()
]

# Create interactive plot
widgets.interact(
    plotting_utils.plot_waveform_with_spectrogram,
    hop_length=(32, 1024, 32),
    n_fft=(32, 2048, 32),
    window=plotting_utils.WINDOW_OPTIONS,
    file=examples,
    cmap=plotting_utils.COLORMAPS,
    speed=[
        ("x1", 1),
        ("x1.5", 1.5),
        ("x2", 2),
        ("x0.5", 0.5),
        ("x0.2", 0.2),
        ("x0.1", 0.1),
    ],
)

Try changing the parameters and see how they affect sounds from different species.

---

🖌️ Can you see that some choice of parameters are good for some species but not for others?

Some species have slowly changing frequency, like a Sheep, hence selecting a high `hop_length` would be able to capture its vocalization accurately without few temporal samples.

Other species, such as the Little Grebe, have quickly varying frequencies, and a high `hop_length` would blur the intricacies of its song.

A small `window_length` can be chosen in case identification does not rely on accurate frequency information.
For example the Hoopoe call consists of a burst of three rapid pulses at low frequencies.
This pattern can be cleary distinguished even with low frequency resolution.

Often species will call at similar frequency bands.
In such case it's best to select a `window_length` that will produce enough frequency resolution to distinguish between similar calls.

---

🖌️ How do the parameters affect the computation time and resulting image size?

- Larger `window_length` will produce taller spectrograms and slow down computation time.
- Smaller `hop_length` will produce lengthier spectrograms and slow down computation time.

In [ ]:
# take a single file
species, filepath = examples[0]

print(
    f"Will generate multiple spectrograms of {species} sounds. Using the file: {filepath.name}"
)

# load the audio
wav, sr = librosa.load(filepath, sr=None)

# select multiple choices of window_length and hop_length
window_lengths = np.arange(64, 2048, 64)
hop_lengths = np.arange(32, 1024, 32)

# create list in which to store the resulting computation times
computation_times = []

# iterate over all window_length and hop_length options
for window_length in window_lengths:
    for hop_length in hop_lengths:
        # start counter
        computation_time = perf_counter()

        # compute spectrogram
        spectrogram = librosa.amplitude_to_db(
            np.abs(
                librosa.stft(
                    wav,
                    hop_length=hop_length,
                    n_fft=window_length,
                    window="hann",
                )
            ),
            ref=np.max,
        )

        # end counter
        computation_time = perf_counter() - computation_time

        # store result in computation_times list
        computation_times.append(
            {
                "window_length": window_length,
                "hop_length": hop_length,
                "computation_time": computation_time,
                "spectrogram_size": spectrogram.size,
            }
        )

# convert computation_times list into a pandas dataframe
computation_times = pd.DataFrame(computation_times)

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.scatterplot(
    data=computation_times,
    x="computation_time",
    y="spectrogram_size",
    size="window_length",
    hue="hop_length",
    sizes=(20, 200),
)
ax.set_xscale("log")
ax.set_yscale("log")

Notice that although spectrogram computation is generally fast, the differences in processing time between settings can exceed an order of magnitude.
When scaling up to millions of recordings, a tenfold increase in computation time becomes highly significant.

Similarly, the size of the resulting spectrograms can vary by up to two orders of magnitude.
Because spectrograms are often used as inputs to deep learning models, their dimensions directly influence model size: larger inputs require models with more parameters, which in turn demand greater computational resources and can make training more difficult.

---

Here we have chosen to use the spectrogram as a visual representation of sound.
Spectrograms are extremely useful transformations, both from a physical and biological perspective, since they capture how energy is distributed across frequencies over time.
However, they represent only one of many possible ways to encode audio information.

When developing a deep learning model for audio (or for any type of data) it is important to consider alternative ways of generating representations or front-ends (so called because they sit at the beginning of the deep learning pipeline).

In audio analysis, for instance, one could provide the model with the raw waveform (a long sequence of amplitude values) or use other transformations that resemble a spectrogram but use a logarithmic frequency scale on the y-axis.
Each representation captures different aspects of the signal and may be more or less suitable depending on the task.

In general, this is one place where domain expertise can make a difference: you can use your understanding of the sound source to choose a representation that may be particularly informative for the task, or alternatively, you can rely on the model itself to learn useful features directly from the raw data.

**🛑 Reflection**

Pause here and reflect on the following questions about your own research.

1. What data are you planning to work with?
2. How is your data represented?
   Are there alternative representations worth considering before starting your machine learning workflow?
3. What characteristics or features of your data might be important to emphasize when choosing a representation?
4. What practical factors (e.g., processing speed, memory usage, or dataset size) might influence your choices or the feasibility of your approach?

## 3. Sound Event Detection

### 3.1 Detecting Sounds

As you can imagine, it is not easy to manually annotate all relevant sounds in a recording.

Take a look at this recording:

In [ ]:
YUCATAN_METADATA = DATA_DIR / "yucatan" / "yucatan_metadata.csv"

YUCATAN_AUDIO_DIR = DATA_DIR / "yucatan" / "audio"

# Load metadata of dataset of bat recordings from the Yucatan peninsula
yucatan = pd.read_csv(YUCATAN_METADATA)

In [ ]:
crowded_recording = YUCATAN_AUDIO_DIR / yucatan.id[1017]

plotting_utils.plot_spectrogram(
    crowded_recording,
    hop_length=128,
    n_fft=512,
    figsize=(14, 4),
)

There are many bat calls in this recording, and it would be very time-consuming to annotate them all.

In [ ]:
empty_recording = YUCATAN_AUDIO_DIR / yucatan.id[205]
plotting_utils.plot_spectrogram(
    empty_recording, hop_length=128, n_fft=512, figsize=(14, 4)
)

This other recording has a single bat pulse.
You still need to review it thoroughly to make sure there are no other sounds of interest.

---

🖌️ Let's briefly think about annotation.
Imagine you are developing a model to automatically detect bats in passive acoustic recordings, though the same ideas apply to many other machine learning tasks in research.

You might already have a set of recordings and a team of experts (perhaps just yourself) who can identify bat species acoustically in your region.
How would you set up an annotation project and provide clear instructions to ensure that annotations are consistent and aligned with your machine learning goals?

In general, you should consider the following when designing annotation guidelines for your team:

1. What counts as a target sound?
   Clearly define the type of sound or event you want to annotate (e.g. bat echolocation calls, social calls, or background noise).

2. How should each target sound be annotated?
   Decide on the level of granularity—are annotators labelling presence/absence, or drawing precise boxes around each call?
   The choice affects both effort and the usefulness of the annotations.

3. What defines a complete annotation?
   Specify the criteria that determine when a recording or segment is fully annotated.

4. How should ambiguous cases be handled?
   Ambiguities often arise due to unusual sounds or borderline cases.
   Reviewing a diverse sample of recordings can help preempt confusion.
   Plan how to document or resolve ambiguous examples.

5. How can you ensure consistency across annotators and sessions?
   Consider strategies such as shared examples, calibration sessions, or inter-annotator agreement checks.

---

### 3.2 Available detection tools

There are some tools that automatically detect animal sounds in recordings.
Various models have been developed and trained for different taxon groups.
The following are very popular examples, all using deep learning:

#### Birds

**BirdNET:** A convolutional neural network from Cornell University for classifying over 5,000 bird calls in audio recordings.

> Kahl, S., Wood, C.M., Eibl, M. and Klinck, H., 2021. BirdNET: A deep learning solution for avian
> diversity monitoring. Ecological Informatics, 61, p.101236.
> [https://www.sciencedirect.com/science/article/pii/S1574954121000273](https://www.sciencedirect.com/science/article/pii/S1574954121000273)

[GitHub](https://github.com/kahst/BirdNET-Analyzer)

**Perch:** Similar idea; model trained on over 10,000 species.
[GitHub](https://github.com/google-research/perch)

#### Bats

**BatDetect2:** A framework for simultaneous detection and classification of UK bats.

> Aodha, O.M., Martínez Balvanera, S., Damstra, E., Cooke, M., Eichinski, P., Browning, E.,
> Barataud, M., Boughey, K., Coles, R., Giacomini, G. and Swiney G, M.C.M., 2022. Towards a general
> approach for bat echolocation detection and classification. bioRxiv, pp.2022-12.
> [https://www.biorxiv.org/content/10.1101/2022.12.14.520490v1.abstract](https://www.biorxiv.org/content/10.1101/2022.12.14.520490v1.abstract)

[GitHub](https://github.com/macaodha/batdetect2)

#### Other

A non-exhaustive list of existing bioacoustic models can be found in the documentation for `bacpipe`, an open-source project designed to facilitate model comparison.
You can view the current selection via the bacpipe [available model list](https://github.com/bioacoustic-ai/bacpipe#available-models).

### 3.3 BatDetect2 Model

Below, we will use BatDetect2 on our data.
Although the model was trained on UK bat calls, we can still test its detection performance on the dataset of bats from the Yucatán peninsula.

💡 Make sure to use a GPU runtime for this.
Even so, completion may take a while as we have quite a number of audio recordings to analyse.

💡 Note: While we are using BatDetect2 within Python for this notebook, you can also run the model directly via the command line.
For full details and installation instructions, please check the official [repository](https://github.com/macaodha/batdetect2).

💡 BatDetect2 internally computes spectrograms to detect and classify bat calls, just like we manually did above ([source code](https://github.com/macaodha/batdetect2/blob/2100a3e483116037c57698c72bbe506c604ebd0b/batdetect2/train/audio_dataloader.py#L521)).

**BatDetect2** can predict multiple bounding boxes for each recording.
Each bounding box has a **score**, a predicted species and a confidence score for the species.

Here, we will throw out the predicted species and confidence score, and only use the bounding box score.
The **score** is the model's confidence that the bounding box contains a bat echolocation call.

In [ ]:
# List all files in the yucatan audio directory
files = api.list_audio_files(DATA_DIR / "yucatan" / "audio")

# Iterate over each recording file
batdetect2_predictions = []
for path in tqdm(files):
    # Run batdetect2 over the file
    output = api.process_file(path)

    # Collect the info we need from each model detection
    pred_dict = output["pred_dict"]
    for detection in pred_dict["annotation"]:
        batdetect2_predictions.append(
            {
                "recording_id": pred_dict["id"],
                "det_prob": detection["det_prob"],  # bbox score only
                "start_time": detection["start_time"],
                "end_time": detection["end_time"],
                "low_freq": detection["low_freq"],
                "high_freq": detection["high_freq"],
            }
        )

# And convert them into a single dataframe
batdetect2_predictions = pd.DataFrame(batdetect2_predictions)

# show the first few rows of predictions made by BatDetect2
batdetect2_predictions.head()

Are these good predictions?
To find out, we need to compare them to ground truth.

Let us load some previously made ground truth annotations first.

In [ ]:
# Load the annotations file
yucatan_annotations = pd.read_csv(DATA_DIR / "yucatan" / "yucatan_annotations.csv")

# show the first few rows
yucatan_annotations.head()

In [ ]:
yucatan_annotations["event"].value_counts()

In [ ]:
pd.crosstab(yucatan_annotations["event"], yucatan_annotations["class"]).T

Now, we can compare the detections with the ground truth.
We can use the Intersection-over-Union (IoU) to measure the overlap between the detections and the ground truth.

<img alt="intersection over union"
src="https://upload.wikimedia.org/wikipedia/commons/c/c7/Intersection_over_Union_-_visual_equation.png"
width="400"></img>

Intuitively, the relationship between overlap (intersection) and union tells us how geometrically similar two bounding boxes are.
An IoU of 1 means that both boxes are perfectly overlapping, for IoU 0 the two boxes don't even touch each other.

We usually cannot expect a machine learning model to always produce perfectly identical bounding boxes as provided in the ground truth.
Hence, we need to set a minimum IoU value for which we count a prediction as "correct".
We will use such an _IoU threshold_ of 0.2 below.

In [ ]:
# Select the predictions and annotations from the crowded recording
file_detections = batdetect2_predictions[
    batdetect2_predictions.recording_id == crowded_recording.name
]
file_annotations = yucatan_annotations[
    yucatan_annotations.recording_id == crowded_recording.name
]

# Match the bounding boxes by computing the IoU. Discard all matches with IoU less than 0.2
pred_boxes = evaluation_utils.bboxes_from_annotations(file_detections)
true_boxes = evaluation_utils.bboxes_from_annotations(file_annotations)
matches = evaluation_utils.match_bboxes(true_boxes, pred_boxes, iou_threshold=0.2)

We can now distinguish three different cases (remember from the previous notebook):

- **True Positives (TP)**: _detections_ with IoU >= threshold
- **False Positives (FP)**: _detections_ with IoU < threshold
- **False Negatives (FN)**: _annotations_ with no matching detection

In [ ]:
# total number of annotated sound events
positives = len(file_annotations)

num_predictions = len(file_detections)

# number of matched prediction boxes
true_positives = len(matches)

# number of predicted boxes that were not matched
false_positives = num_predictions - len(matches)

# number of annotated sound events that were not matched
false_negatives = positives - len(matches)

With this information, we can compute the precision and recall of the detections:

$$ \text{Precision} = \frac{\text{TP}}{\text{TP + FP}} $$

$$ \text{Recall} = \frac{\text{TP}}{\text{TP + FN}} $$

In [ ]:
# Percentage of predictions that are correct
precision = true_positives / num_predictions

# Percentage of sound events that were detected
recall = true_positives / positives

print(
    f"BatDetect2 precision={precision:.1%} recall={recall:.1%} on file {crowded_recording.name}"
)

Now, we can visualise predictions and ground truth annotations in the spectrograms together:

In [ ]:
score_threshold = 0.2
iou_threshold = 0.2

plotting_utils.plot_spectrogram_with_predictions_and_annotations(
    crowded_recording,
    batdetect2_predictions[batdetect2_predictions.det_prob > score_threshold],
    yucatan_annotations,
    iou_threshold=iou_threshold,
)

In the visualisation above, bounding boxes are drawn in different styles:

- red = spurious predicted sound event (false positive)
- green = correct prediction (true positive)
- white = missed sound event (false negative)

Besides an IoU threshold, you will notice that we also need to set a _confidence threshold_, because as explained above, BatDetect2 provides a confidence score.

We have set this rather arbitrarily to 0.3 above.
You can imagine what happens if we increase or decrease this threshold.
Watch what happens for different files when you adjust IoU and confidence thresholds in the widget below.

In [ ]:
# @title Batdetect2 predictions

# @markdown Select a file and an IoU threshold.

example_files = [
    YUCATAN_AUDIO_DIR / row["id"] for _, row in yucatan.sample(n=20).iterrows()
]


@widgets.interact(
    path=[(path.name, path) for path in example_files],
    iou_threshold=(0, 1, 0.025),
    score_threshold=(0, 1, 0.025),
)
def plot_batdetect2_results_file_results(
    path=crowded_recording,
    iou_threshold=0.5,
    score_threshold=0.3,
):
    confident_detections = batdetect2_predictions[
        batdetect2_predictions.det_prob > score_threshold
    ]

    precision, recall = evaluation_utils.compute_file_precision_recall(
        path,
        confident_detections,
        yucatan_annotations,
        iou_threshold=iou_threshold,
    )

    print(
        f"Batdetect2 precision={precision:.1%} recall={recall:.1%} on file {path.name}"
    )

    plotting_utils.plot_spectrogram_with_predictions_and_annotations(
        path,
        confident_detections,
        yucatan_annotations,
        iou_threshold=iou_threshold,
        linewidth=1,
    )

---

🖌️ Run the code below to calculate the following properties:

- The mean precision and recall across all files.
- The percentage of files where all bat calls were missed.

In [ ]:
precisions, recalls = [], []  # lists of per-file precision and recall scores
files_missed = 0  # counter for number of files where all calls were missed

for recording_id in batdetect2_predictions.recording_id.unique():
    file_detections = batdetect2_predictions[
        batdetect2_predictions.recording_id == recording_id
    ]
    file_annotations = yucatan_annotations[
        yucatan_annotations.recording_id == recording_id
    ]

    # Match the bounding boxes by computing the IoU. Discard all matches with IoU less than 0.2
    pred_boxes = evaluation_utils.bboxes_from_annotations(file_detections)
    true_boxes = evaluation_utils.bboxes_from_annotations(file_annotations)
    matches = evaluation_utils.match_bboxes(true_boxes, pred_boxes, iou_threshold=0.2)

    # total number of annotated sound events
    positives = len(file_annotations)

    num_predictions = len(file_detections)

    # number of matched prediction boxes
    true_positives = len(matches)

    # number of predicted boxes that were not matched
    false_positives = num_predictions - len(matches)

    # number of annotated sound events that were not matched
    false_negatives = positives - len(matches)

    # Percentage of predictions that are correct
    if num_predictions == 0:
        precision = 0
    else:
        precision = true_positives / num_predictions

    # Percentage of sound events that were detected
    if positives == 0:
        recall = 0
    else:
        recall = true_positives / positives

    precisions.append(precision)
    recalls.append(recall)

    # increment counter if all bat calls were missed
    if true_positives == 0 and positives > 0:
        files_missed += 1

print(
    f"Mean precision = {np.mean(precisions):.2%}, Mean recall = {np.mean(recalls):.2%}."
)
print(
    f"Percent of files where all bat calls were missed = {files_missed / len(precisions):.2%}."
)

---

🖌️ Run the full evaluation again but change the `iou_threshold` parameter.
What do you observe?

- Increasing the **IoU threshold** makes it harder to find matches (`true_positives`).
  Recall will necessarily be lowered, as less bat calls can be detected.
  Precision will also be lowered as less detections can be correct.

- Conversely, a lower **IoU threshold** increases the number of matches (`true_positives`), making both recall and precision greater.

- Selecting an **IoU threshold** is not about optimising performance, but about choosing a matching criterion.
  Having a low **IoU** threshold might increase the recall and precision but incurr a cost in precision of the bounding box of each detection.

We can visualise BatDetect2's predictions versus ground truth labels for a selection of audio files below.

In [ ]:
# @title Batdetect2 predictions

# @markdown Select a file and an IoU threshold.

example_files = [
    YUCATAN_AUDIO_DIR / row["id"] for _, row in yucatan.sample(n=20).iterrows()
]


@widgets.interact(
    path=example_files,
    iou_threshold=(0, 1, 0.05),
    score_threshold=(0, 1, 0.05),
)
def plot_batdetect2_results_file_results(
    path=crowded_recording,
    iou_threshold=0.2,
    score_threshold=0.3,
):
    confident_detections = batdetect2_predictions[
        batdetect2_predictions.det_prob > score_threshold
    ]

    precision, recall = evaluation_utils.compute_file_precision_recall(
        path,
        confident_detections,
        yucatan_annotations,
        iou_threshold=iou_threshold,
    )

    print(
        f"Batdetect2 precision={precision:.1%} recall={recall:.1%} on file {path.name}"
    )

    plotting_utils.plot_spectrogram_with_predictions_and_annotations(
        path,
        confident_detections,
        yucatan_annotations,
        iou_threshold=iou_threshold,
        linewidth=2,
    )

---

**🛑 Reflection**

In this section, we explored how to evaluate the performance of BatDetect2 in identifying echolocation pulses using metrics such precision, and recall.
These evaluation metrics summarise model performance across many examples, and each one captures only a specific aspect of that behavior.
For example, we did not evaluate how closely the predicted bounding boxes align with expert annotations.

Take a moment to think on the following questions:

1. For the machine learning tasks in your research, which evaluation metrics are available, and which ones are most commonly used?
2. Do these metrics align with your broader research goals?
   Metrics are often adopted because they are standard in ML research, but they may not fully reflect, or provide a holistic view of, your ecological or scientific objectives.
3. Who are the potential users of your research outputs?
   If you've identified them, what information would they need to make informed decisions about how and when to use your results?

---

Notice that we only evaluated how well the model detects general echolocation calls, regardless of species.
This same method is used to measure performance for specific target species.
The evaluation runs independently for each target sound, and you can average the results to get an overall summary of model performance across all species.
Hopefully, this leaves you better equipped to interpret outputs from other models and taxonomic groups in the future.

## Lunch Break

We will now pause for lunch.
When we return, we will focus on deploying BatDetect2 and BirdNET onto an edge device.